<a href="https://colab.research.google.com/github/josephgalicinao/SkinLesionDetection/blob/main/Skin_Lesion_Segmentation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Initialization

## Imports

Important imports and packages used

In [ ]:
from google.colab import userdata
import os
import kagglehub
import pandas as pd
import tensorflow as tf
import os
import csv
import numpy as np
import matplotlib.pyplot as plt
import cv2
from sklearn.model_selection import train_test_split
from sklearn.model_selection import StratifiedGroupKFold

import csv
import pandas as pd
from sklearn.preprocessing import StandardScaler
from tqdm import tqdm

IMG_SIZE   = (128, 128)
AUTOTUNE   = tf.data.AUTOTUNE

## Load Dataset

In [ ]:
os.environ["KAGGLE_USERNAME"] =  userdata.get('KAGGLE_USERNAME')
os.environ["KAGGLE_KEY"] = userdata.get('KAGGLE_KEY')

!pip install -q kaggle

Loads the HAM10000 dataset

In [ ]:
ham10000_dataset_path = kagglehub.dataset_download("josephgalicinao/skin-lesion-dataset")

print("Path to dataset files:", ham10000_dataset_path)

100%|██████████| 2.90G/2.90G [03:21<00:00, 15.4MB/s]

Extracting files...


Path to dataset files: /root/.cache/kagglehub/datasets/josephgalicinao/skin-lesion-dataset/versions/1


## Create Datasets

Load data and masks

In [ ]:
def get_HAM10000_paths(image_ids):
    '''
    Gets the paths of each image, mask, and distance map based on the image ID
    Makes sure that the image, mask, and distance map exist
    image_ids: list of image IDs
    returns: img_paths, mask_paths, distance_paths
    '''

    img_paths = []
    mask_paths = []
    distance_paths = []
    for img_id in image_ids:
        mask_path = os.path.join(f"{ham10000_dataset_path}/HAM10000_masks/", f"{img_id}_segmentation.png")
        image_path = os.path.join(f"{ham10000_dataset_path}/HAM10000_images", f"{img_id}.jpg")
        distance_path = os.path.join(f"{ham10000_dataset_path}/HAM10000_distance_maps", f"{img_id}_distance_map.png")
        if os.path.exists(mask_path) and os.path.exists(image_path) and os.path.exists(distance_path):
          mask_paths.append(mask_path)
          img_paths.append(image_path)
          distance_paths.append(distance_path)

    assert len(img_paths) == len(mask_paths) == len(distance_paths) == len(image_ids)

    return img_paths, mask_paths, distance_paths

def load_HAM10000():
  '''
  Returns the image IDs to load images
  Returns the labels for the images
  Returns the patient IDs for the images to avoid data leakage
  '''

  # HAM10000 Dataset
  ham10000_df = pd.read_csv(f"{ham10000_dataset_path}/HAM10000_metadata.csv")

  # Get all the different patient labels
  patient_labels = ham10000_df.groupby("lesion_id")["dx"].first().reset_index()

  # Split based on unique skin lesion IDs
  # Stratify based on the skin lesion type (dx) to have equal balance size
  train_patients, test_patients = train_test_split(
      patient_labels["lesion_id"].values, test_size=0.2, stratify=patient_labels["dx"].values, random_state=42
  )

  # Get all the training data
  train_df = ham10000_df[ham10000_df["lesion_id"].isin(train_patients)]

  train_image_ids = train_df["image_id"].values
  train_dx = train_df["dx"].values
  train_lesion_ids = train_df["lesion_id"].values

  test_df = ham10000_df[ham10000_df["lesion_id"].isin(test_patients)]
  test_image_ids = test_df["image_id"].values
  test_dx = test_df["dx"].values
  test_lesion_ids = test_df["lesion_id"].values

  return train_image_ids, train_dx, train_lesion_ids, test_image_ids, test_dx, test_lesion_ids

Prepares the dataset for use in TF

In [ ]:
def segmentation_load_dataset(img_path, mask_path, distance_path):
    '''
    Loads the image, mask, and distance maps
    img_path: path to image
    mask_path: path to mask
    distance_path: path to distance map
    returns: image, mask, distance map
    '''
    img = tf.io.read_file(img_path)
    img = tf.image.decode_jpeg(img, channels=3)       # decode JPG as RGB (3 channels)
    img = tf.image.resize(img, IMG_SIZE)
    img = tf.cast(img, tf.float32) / 255.0
    img.set_shape([IMG_SIZE[0], IMG_SIZE[1], 3])

    # MASK (PNG, already grayscale)
    m = tf.io.read_file(mask_path)
    m = tf.image.decode_png(m, channels=1)
    m = tf.image.resize(m, IMG_SIZE, method='nearest')
    m = tf.cast(m > 127, tf.float32)                  # binarize mask (0 or 1)
    m.set_shape([IMG_SIZE[0], IMG_SIZE[1], 1])

    # DISTANCE MAP
    d = tf.io.read_file(distance_path)
    d = tf.image.decode_png(d, channels=1)
    d = tf.image.resize(d, IMG_SIZE, method='nearest')
    d = tf.cast(d, tf.float32) / 255.0
    d = d + 1
    d.set_shape([IMG_SIZE[0], IMG_SIZE[1], 1])

    return img, m, d

def segmentation_augment(img, mask, dist):
    # Random horizontal flip
    if tf.random.uniform(()) > 0.5:
        img  = tf.image.flip_left_right(img)
        mask = tf.image.flip_left_right(mask)
        dist = tf.image.flip_left_right(dist)

    if tf.random.uniform(()) > 0.5:
        img  = tf.image.flip_up_down(img)
        mask = tf.image.flip_up_down(mask)
        dist = tf.image.flip_up_down(dist)

    if tf.random.uniform(()) > 0.5:
        img  = tf.image.random_brightness(img, max_delta=0.1)

    if tf.random.uniform(()) > 0.5:
        img  = tf.image.random_contrast(img, lower=0.9, upper=1.1)

    if tf.random.uniform(()) > 0.5:
        img = tf.image.rot90(img)
        mask  = tf.image.rot90(mask)
        dist  = tf.image.rot90(dist)

    return img, mask, dist

def format_deep_supervision(image, mask, dist):
    # Downsample the mask for each decoder block
    m2 = tf.image.resize(mask, (64, 64), method='nearest')
    m3 = tf.image.resize(mask, (32, 32), method='nearest')
    m4 = tf.image.resize(mask, (16, 16), method='nearest')

    masks = {
        "o1": mask,  # 128x128
        "o2": m2,    # 64x64
        "o3": m3,    # 32x32
        "o4": m4,    # 16x16
    }

    dist2 = tf.image.resize(dist, (64, 64), method='nearest')
    dist3 = tf.image.resize(dist, (32, 32), method='nearest')
    dist4 = tf.image.resize(dist, (16, 16), method='nearest')

    dists = {
        "o1": dist,  # 128x128
        "o2": dist2, # 64x64
        "o3": dist3, # 32x32
        "o4": dist4, # 16x16
    }

    return image, masks, dists

Make the straifier and k fold

In [ ]:
# Stratifier
sgkf = StratifiedGroupKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

# Segmentation

## Loss Functions

In [ ]:
def dice_loss(y_true, y_pred, smooth=1e-6):
    intersection = tf.reduce_sum(y_true * y_pred, axis=[1, 2, 3])
    sums = tf.reduce_sum(y_true, axis=[1, 2, 3]) + tf.reduce_sum(y_pred, axis=[1, 2, 3])
    dice = (2.0 * intersection + smooth) / (sums + smooth)
    return 1.0 - tf.reduce_mean(dice)

def iou_loss(y_true, y_pred, smooth=1e-6):
    intersection = tf.reduce_sum(y_true * y_pred, axis=[1,2,3])  # sum over H, W, C
    union = tf.reduce_sum(y_true, axis=[1,2,3]) + tf.reduce_sum(y_pred, axis=[1,2,3]) - intersection
    iou = (intersection + smooth) / (union + smooth)
    return 1.0 - tf.reduce_mean(iou)  # average IoU across batch

def bce_iou_loss(y_true, y_pred):
    bce = tf.reduce_mean(tf.keras.losses.binary_crossentropy(y_true, y_pred))
    iou = iou_loss(y_true, y_pred)
    beta = 0.25
    return beta * bce + (1 - beta) * iou

def bce_dice_loss(y_true, y_pred):
    bce = tf.reduce_mean(tf.keras.losses.binary_crossentropy(y_true, y_pred))
    dice = dice_loss(y_true, y_pred)
    return bce + tf.math.log(tf.math.cosh(dice))

TF evaluation metrics that will be used to monitor the IoU and dice while training

In [ ]:
def hard_dice(y_true, y_pred, threshold=0.5, smooth=1e-6):
    # Ensure float32
    y_true = tf.cast(y_true, tf.float32)
    y_pred = tf.cast(y_pred, tf.float32)

    # Apply threshold to predictions
    y_pred = tf.cast(y_pred > threshold, tf.float32)

    # Compute intersection and union per image
    intersection = tf.reduce_sum(y_true * y_pred, axis=[1,2,3])
    union = tf.reduce_sum(y_true, axis=[1,2,3]) + tf.reduce_sum(y_pred, axis=[1,2,3])

    # Dice coefficient per image
    dice = (2 * intersection + smooth) / (union + smooth)

    # Return mean Dice over batch
    return tf.reduce_mean(dice)

def hard_iou(y_true, y_pred, threshold=0.5, smooth=1e-6):
    # Ensure float32
    y_true = tf.cast(y_true, tf.float32)
    y_pred = tf.cast(y_pred, tf.float32)

    # Apply threshold to predictions
    y_pred = tf.cast(y_pred > threshold, tf.float32)

    # Compute intersection and union per image
    intersection = tf.reduce_sum(y_true * y_pred, axis=[1,2,3])
    union = tf.reduce_sum(y_true, axis=[1,2,3]) + tf.reduce_sum(y_pred, axis=[1,2,3]) - intersection

    # IoU per image
    iou = (intersection + smooth) / (union + smooth)

    # Return mean IoU over batch
    return tf.reduce_mean(iou)

Numpy evaluation metrics used for getting the IoU and dice for segmented masks

In [ ]:
def dice_np(y_true, y_pred, threshold=0.5, smooth=1e-7):
    y_true = (y_true > threshold)
    y_pred = (y_pred > threshold)

    intersection = np.logical_and(y_true, y_pred).sum()
    return (2. * intersection + smooth) / (y_true.sum() + y_pred.sum() + smooth)


def iou_np(y_true, y_pred, threshold=0.5, smooth=1e-7):
    y_true = (y_true > threshold)
    y_pred = (y_pred > threshold)

    intersection = np.logical_and(y_true, y_pred).sum()
    union = np.logical_or(y_true, y_pred).sum()
    return (intersection + smooth) / (union + smooth)

## Segmentation Training Pipeline

General training pipeline for training segmentation

In [ ]:
def train_segmentation_model(model, train_ds, val_ds, loss="binary_crossentropy", learning_rate=0.0001, epochs=20, deep_supervision=False):

  reduce_lr = tf.keras.callbacks.ReduceLROnPlateau(
    monitor='o1_hard_iou',  # Metric to monitor (e.g., 'val_accuracy' or 'loss')
    factor=0.5,          # Factor by which the learning rate will be reduced (new_lr = lr * factor)
    patience=3,          # Number of epochs with no improvement after which learning rate will be reduced
    min_delta=0.0001,    # Threshold for measuring the new optimum, to focus only on significant changes
    cooldown=0,          # Number of epochs to wait before resuming normal operation after lr has been reduced
    min_lr=0.000001,      # Lower bound on the learning rate
    verbose=1,            # 0: quiet, 1: update messages
    mode='max'           # One of {
  )

  # Create optimizer
  optimizer = tf.keras.optimizers.Adam(learning_rate=learning_rate)

  # Create early stopping
  early_stopping = tf.keras.callbacks.EarlyStopping(
      monitor='o1_hard_iou',
      min_delta=0.001,
      patience=9,
      verbose=1, # Prints a message when stopping
      mode='max',
      restore_best_weights=True
  )

  if deep_supervision:
    metrics = {"o1": [hard_dice, hard_iou]}
    loss_func = {"o1": loss,
                 "o2": loss,
                 "o3": loss,
                 "o4": loss}
  else:
    metrics = [hard_dice, hard_iou]
    loss_func = loss

  # Compile the model
  model.compile(
      optimizer=optimizer,
      loss=loss_func,
      metrics=metrics
  )

  # Fit the model
  model.fit(
      train_ds,
      validation_data=val_ds,
      epochs=epochs,
      callbacks=[early_stopping, reduce_lr]
  )

  print(f"Loss: {loss}")

  return model

## U-Net

### Initialize U-Net

In [ ]:
def unet_encoder_block(inputs, num_filters, residual=False, activation="relu"):
    x = tf.keras.layers.Conv2D(num_filters, 3, padding="same")(inputs)
    x = tf.keras.layers.BatchNormalization()(x)
    x = tf.keras.layers.Activation(activation)(x)
    x = tf.keras.layers.Conv2D(num_filters, 3, padding="same")(x)
    x = tf.keras.layers.BatchNormalization()(x)

    if residual:
      # Make the input the same size as x so we can add them together
      shortcut = tf.keras.layers.Conv2D(num_filters, 1, strides=1, padding='same')(inputs)
      x = tf.keras.layers.Add()([x, shortcut])

    x = tf.keras.layers.Activation(activation)(x)

    return x

def attention_gate(x, g, inter_channels, activation="relu"):
    """
    x: skip connection (encoder feature map)
    g: gating signal (decoder feature map, upsampled)
    """

    # Project skip and gate to lower-dimensional space
    theta_x = tf.keras.layers.Conv2D(inter_channels, 1, padding="same")(x)
    theta_x = tf.keras.layers.BatchNormalization()(theta_x)

    phi_g   = tf.keras.layers.Conv2D(inter_channels, 1, padding="same")(g)
    phi_g   = tf.keras.layers.BatchNormalization()(phi_g)

    # Combine
    f = tf.keras.layers.Add()([theta_x, phi_g])
    f = tf.keras.layers.Activation(activation)(f)

    # Produce attention coefficients
    psi = tf.keras.layers.Conv2D(1, 1, padding="same")(f)
    psi = tf.keras.layers.Activation("sigmoid")(psi)

    # Apply attention to skip connection
    out = tf.keras.layers.Multiply()([x, psi])

    return out

def unet_decoder_block(inputs, skip_feature, num_filters, residual=False, attention=False, activation="relu"):
    upsampled = tf.keras.layers.Conv2DTranspose(num_filters, 2, strides=2, padding="same")(inputs)

    if attention:
      skip_feature = attention_gate(
          x=skip_feature,
          g=upsampled,
          inter_channels=num_filters // 2
      )

    # Concatenate
    concatenate = tf.keras.layers.Concatenate()([upsampled, skip_feature])

    x = tf.keras.layers.Conv2D(num_filters, 3, padding="same")(concatenate)
    x = tf.keras.layers.BatchNormalization()(x)
    x = tf.keras.layers.Activation(activation)(x)
    x = tf.keras.layers.Conv2D(num_filters, 3, padding="same")(x)
    x = tf.keras.layers.BatchNormalization()(x)

    if residual:
      shortcut = tf.keras.layers.Conv2D(num_filters, 1, strides=1, padding='same')(concatenate)
      x = tf.keras.layers.Add()([x, shortcut])

    x = tf.keras.layers.Activation(activation)(x)

    return x

def unet_model(input_shape=(512, 512, 3), num_classes=1, deep_supervision=False, attention=False, residual=False, activation="relu"):
    inputs = tf.keras.layers.Input(shape=input_shape)

    e1 = unet_encoder_block(inputs, 64, residual=residual, activation=activation)
    e2 = unet_encoder_block(tf.keras.layers.MaxPool2D(pool_size=(2, 2), strides=2)(e1), 128, residual=residual, activation=activation)
    e3 = unet_encoder_block(tf.keras.layers.MaxPool2D(pool_size=(2, 2), strides=2)(e2), 256, residual=residual, activation=activation)
    e4 = unet_encoder_block(tf.keras.layers.MaxPool2D(pool_size=(2, 2), strides=2)(e3), 512, residual=residual, activation=activation)

    b = unet_encoder_block(tf.keras.layers.MaxPool2D(pool_size=(2, 2), strides=2)(e4), 1024, residual=residual, activation=activation)

    d4 = unet_decoder_block(b, e4, 512, residual=residual, attention=attention, activation=activation)
    d3 = unet_decoder_block(d4, e3, 256, residual=residual, attention=attention, activation=activation)
    d2 = unet_decoder_block(d3, e2, 128, residual=residual, attention=attention, activation=activation)
    d1 = unet_decoder_block(d2, e1, 64, residual=residual, attention=attention, activation=activation)

    outputs = tf.keras.layers.Conv2D(num_classes, 1, padding='same', activation='sigmoid')(d1)

    # For deep supervision
    if deep_supervision:
      o1 = tf.keras.layers.Conv2D(num_classes, 1, padding='same', activation='sigmoid', name="o1")(d1)
      o2 = tf.keras.layers.Conv2D(num_classes, 1, padding='same', activation='sigmoid', name="o2")(d2)
      o3 = tf.keras.layers.Conv2D(num_classes, 1, padding='same', activation='sigmoid', name="o3")(d3)
      o4 = tf.keras.layers.Conv2D(num_classes, 1, padding='same', activation='sigmoid', name="o4")(d4)

      outputs = {
          "o1": o1,
          "o2": o2,
          "o3": o3,
          "o4": o4,
      }

    model = tf.keras.models.Model(inputs=inputs, outputs=outputs, name="U-Net")
    print(f"Residual: {residual}")
    print(f"Attention: {attention}")
    print(f"Deep Supervision: {deep_supervision}")
    print(f"Activation: {activation}")
    return model

unet_model(input_shape=(128, 128, 3), num_classes=1, deep_supervision=True, residual=False, attention=True)

Residual: False
Attention: True
Deep Supervision: True
Activation: relu


<Functional name=U-Net, built=True>

### Train U-net

In [ ]:
# Parameters to change
deep_supervision = True
residual = True
attention = True
activation = "swish"
batch = 64
loss = bce_iou_loss

image_ids, dx, lesion_ids = load_HAM10000()

with open('/content/drive/MyDrive/Thesis/segmentation_activation.csv', 'w', newline='') as csvfile:
  csv_writer = csv.writer(csvfile)
  csv_writer.writerow(["LR", "Val Dice", "Val IoU"])

  for fold, (train_idx, val_idx) in enumerate(sgkf.split(image_ids, dx, lesion_ids)):
      print(f"\n===== Fold {fold+1} =====")

      # Gets the image ID for image paths
      train_ids, val_ids = image_ids[train_idx], image_ids[val_idx]

      # Get the corresponding image and train masks
      train_imgs, train_masks, train_dist = get_paths(train_ids)
      val_imgs, val_masks, val_dist = get_paths(val_ids)

      # Get the training and validation datasets
      train_ds = (
          tf.data.Dataset.from_tensor_slices((train_imgs, train_masks, train_dist))
          .map(load_image_and_mask, num_parallel_calls=AUTOTUNE)
          .map(normalize_image, num_parallel_calls=AUTOTUNE)
          .map(augment, num_parallel_calls=AUTOTUNE)
          .map(format_deep_supervision, num_parallel_calls=AUTOTUNE)
          .shuffle(200)
          .batch(batch)
          .prefetch(AUTOTUNE)
      )

      val_ds = (
          tf.data.Dataset.from_tensor_slices((val_imgs, val_masks, val_dist))
          .map(load_image_and_mask, num_parallel_calls=AUTOTUNE)
          .map(normalize_image, num_parallel_calls=AUTOTUNE)
          .map(format_deep_supervision, num_parallel_calls=AUTOTUNE)
          .batch(batch)
          .prefetch(AUTOTUNE)
      )

      # Create the model
      model = unet_model(input_shape=(IMG_SIZE[0], IMG_SIZE[1], 3), num_classes=1,
                            deep_supervision=deep_supervision,
                            attention=attention,
                            residual=residual,
                            activation=activation)

      # Fit the model
      fitted_model = train_segmentation_model(model, train_ds, val_ds, epochs=200, learning_rate=0.0001, deep_supervision=deep_supervision, loss=loss)

      # Save model for future use
      model.save(f'/content/drive/MyDrive/Thesis/segmentation_model_fold{fold + 1}.keras')

      # Get validation segmentation masks based on the fitted model
      validation_masks = fitted_model.predict(val_ds)

      # Get the original masks
      original_images, groundtruth_masks, _ = get_paths(val_ids)

      dice = 0.0
      IoU = 0.0

      for i in range(len(val_ids)):
        # Get the groundtruth mask and normalize between 0-1
        # Color Image

        groundtruth_mask = groundtruth_masks[i]
        groundtruth_mask = cv2.imread(groundtruth_mask, cv2.IMREAD_GRAYSCALE) / 255.0

        # Resize the validation masks
        mask = validation_masks["o1"][i]
        mask = cv2.resize(mask, (600, 450))
        mask = (mask > 0.5).astype("uint8") * 255

        # Calculate IoU and dice
        dice += dice_np(groundtruth_mask, mask)
        IoU += iou_np(groundtruth_mask, mask)

      dice /= len(val_ids)
      IoU /= len(val_ids)

      print(f"Average Dice: {dice}")
      print(f"Average IoU: {IoU}")

      # Train the model and write to a CSV file
      csv_writer.writerow(["swish", dice, IoU])

ValueError: too many values to unpack (expected 3)

Train on all data for final model

In [ ]:
# Parameters to change
deep_supervision = True
residual = True
attention = True
activation = "swish"
batch = 64
loss = bce_iou_loss

train_image_ids, train_dx, train_lesion_ids, test_image_ids, test_dx, test_lesion_ids = load_HAM10000()

# Get the corresponding image and train masks
train_imgs, train_masks, train_dist = get_HAM10000_paths(train_image_ids)
val_imgs, val_masks, val_dist = get_HAM10000_paths(test_image_ids)

# Get the training and validation datasets
train_ds = (
    tf.data.Dataset.from_tensor_slices((train_imgs, train_masks, train_dist))
    .map(segmentation_load_dataset, num_parallel_calls=AUTOTUNE)
    .map(segmentation_augment, num_parallel_calls=AUTOTUNE)
    .map(format_deep_supervision, num_parallel_calls=AUTOTUNE)
    .shuffle(2000)
    .batch(batch)
    .prefetch(AUTOTUNE)
)

val_ds = (
    tf.data.Dataset.from_tensor_slices((val_imgs, val_masks, val_dist))
    .map(segmentation_load_dataset, num_parallel_calls=AUTOTUNE)
    .map(format_deep_supervision, num_parallel_calls=AUTOTUNE)
    .batch(batch)
    .prefetch(AUTOTUNE)
)


# Create the model
model = unet_model(input_shape=(IMG_SIZE[0], IMG_SIZE[1], 3), num_classes=1,
                      deep_supervision=deep_supervision,
                      attention=attention,
                      residual=residual,
                      activation=activation)

# Fit the model
fitted_model = train_segmentation_model(model, train_ds, val_ds=val_ds, epochs=200, learning_rate=0.0001, deep_supervision=deep_supervision, loss=loss)
fitted_model.evaluate(val_ds)
# Save model for future use
# model.save(f'/content/drive/MyDrive/Thesis/segmentation_model.keras')
model.save(f'/content/drive/MyDrive/Thesis/segmentation_model_v2.keras')

Residual: True
Attention: True
Deep Supervision: True
Activation: swish
Epoch 1/200
126/126 ━━━━━━━━━━━━━━━━━━━━ 131s 452ms/step - loss: 1.5998 - o1_hard_dice: 0.8479 - o1_hard_iou: 0.7688 - o1_loss: 0.4372 - o2_loss: 0.3999 - o3_loss: 0.3788 - o4_loss: 0.3816 - val_loss: 6.3650 - val_o1_hard_dice: 2.6567e-04 - val_o1_hard_iou: 1.5288e-04 - val_o1_loss: 1.5786 - val_o2_loss: 1.6137 - val_o3_loss: 1.6097 - val_o4_loss: 1.5989 - learning_rate: 1.0000e-04
Epoch 2/200
126/126 ━━━━━━━━━━━━━━━━━━━━ 21s 144ms/step - loss: 1.0587 - o1_hard_dice: 0.9123 - o1_hard_iou: 0.8527 - o1_loss: 0.2642 - o2_loss: 0.2630 - o3_loss: 0.2615 - o4_loss: 0.2687 - val_loss: 2.3350 - val_o1_hard_dice: 0.8095 - val_o1_hard_iou: 0.7113 - val_o1_loss: 0.5655 - val_o2_loss: 0.5717 - val_o3_loss: 0.5944 - val_o4_loss: 0.6410 - learning_rate: 1.0000e-04
Epoch 3/200
126/126 ━━━━━━━━━━━━━━━━━━━━ 21s 143ms/step - loss: 0.9700 - o1_hard_dice: 0.9199 - o1_hard_iou: 0.8644 - o1_loss: 0.2420 - o2_loss: 0.2409 - o3_loss: 0.24

### Postprocessing

Morphological Operations

In [ ]:
image_ids, dx, lesion_ids = load_HAM10000()

with open('/content/drive/MyDrive/Thesis/unet_gaussian_blue.csv', 'w', newline='') as csvfile:
  csv_writer = csv.writer(csvfile)
  csv_writer.writerow(["LR", "Val Dice", "Val IoU"])

  for fold, (train_idx, val_idx) in enumerate(sgkf.split(image_ids, dx, lesion_ids)):
      print(f"\n===== Fold {fold+1} =====")

      # Gets the image ID for image paths
      _, val_ids = image_ids[train_idx], image_ids[val_idx]

      # Get the corresponding image and train masks
      val_imgs, val_masks, val_dist = get_HAM10000_paths(val_ids)

      val_ds = (
          tf.data.Dataset.from_tensor_slices((val_imgs, val_masks, val_dist))
          .map(load_image_and_mask, num_parallel_calls=AUTOTUNE)
          .map(normalize_image, num_parallel_calls=AUTOTUNE)
          .map(format_deep_supervision, num_parallel_calls=AUTOTUNE)
          .batch(batch)
          .prefetch(AUTOTUNE)
      )

      segmentation_model_path = f'/content/drive/MyDrive/Thesis/Models/ham10000_segmentation_model_fold{fold + 1}.keras'
      segmentation_model = tf.keras.models.load_model(segmentation_model_path,
                                                      custom_objects={"bce_iou_loss": bce_iou_loss,
                                                                      "hard_dice": hard_dice,
                                                                      "hard_iou": hard_iou})

      validation_masks = segmentation_model.predict(val_ds)

      for kernel_size in [3, 7, 11]:
        dice = 0.0
        IoU = 0.0

        for i in range(len(val_ids)):
          # Postpress each mask and get its corresponding feature
          cur_mask = cv2.resize(validation_masks["o1"][i], (600, 450))
          cur_mask = (cur_mask > 0.5).astype(np.uint8) * 255

          # Gaussian blur
          cur_mask = cv2.GaussianBlur(cur_mask, (kernel_size, kernel_size), 0) / 255.0

          groundtruth_mask = cv2.imread(val_masks[i], cv2.IMREAD_GRAYSCALE) / 255.0

          # Calculate dice and IoU
          dice += dice_np(groundtruth_mask, cur_mask)
          IoU += iou_np(groundtruth_mask, cur_mask)

        dice /= len(val_ids)
        IoU /= len(val_ids)

        csv_writer.writerow([kernel_size, dice, IoU])